In [1]:
import numpy as np
import scipy
import utils
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from matplotlib.backends.backend_pdf import PdfPages


In [2]:
def get_atm_center(ds_atm, sfc_center):
    current_level_center = sfc_center
    current_lat = current_level_center['center_lat'].values
    current_lon = current_level_center['center_lon'].values
    center_points = {'Surface_Layer': [],'700mb': [], '200mb': []}
    for i, level in enumerate(ds_atm.isobaricInhPa):
        ds_atm_lvl = ds_atm.sel(latitude = slice(current_lat - 2, current_lat.values + 2),
                        longitude = slice(current_lon - 2, current_lon + 2))
        ds_atm_lvl = ds_atm_lvl.sel(isobaricInhPa = level)
        if ds_atm_lvl['gh'].isnull().any():
            print('NaN values in storm region. Skipping Frame')
            try:
                ds_atm_lvl['gh'].plot()
                plt.show()
            except Exception as e:
                print(e)    
            return None
        
        hgt_threshold = np.quantile(ds_atm_lvl['gh'], 0.01)
        hgt_clusters = ds_atm_lvl['gh'].where(ds_atm_lvl['gh'] < hgt_threshold)
        ght_clusters_mask = ~hgt_clusters.isnull()

        clusters = scipy.ndimage.label(ght_clusters_mask)
        cluster_size = []
        for a in np.arange(clusters[1]):
            cluster_number = a + 1
            cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

        
        largest_cluster = np.argmax(cluster_size) + 1
        center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)

        current_lat = ds_atm_lvl.latitude.isel(latitude =int(np.round(center_y)))
        current_lon = ds_atm_lvl.longitude.isel(longitude =int(np.round(center_x)))

        if i == 0:  
            center_points['Surface_Layer'] = [current_lon, current_lat]
        elif i == 1:
            center_points['700mb'] = [current_lon, current_lat]
        elif i == 2:
            center_points['200mb'] = [current_lon, current_lat]

        return center_points

# def get_sfc_center(ds):
#     min_pressure = ds['prmsl'].min().values

#     ## Threshold is just based on the lowest mslp within the frame
#     ## Given that the calculated center is the center of mass for 
#     ## the largest contingous 1% low pressure area, final center
#     ## pressure and lowest pressure in the frame may not be exactly
#     ## the same.
#     if min_pressure > 1005: # Pressure threshold (pa) 1005mb
#         print(f'{min_pressure}mb is over the pressure threshold')
#         return None
#     nan_mask = ds['prmsl'].isnull()
#     ds_sfc = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    

#     slp_threshold = np.quantile(ds_sfc['prmsl'], 0.01)
#     slp_clusters = ds_sfc['prmsl'].where(ds_sfc['prmsl'] < slp_threshold)
#     slp_clusters_mask = ~slp_clusters.isnull()

#     clusters = scipy.ndimage.label(slp_clusters_mask)
#     cluster_size = []
#     for a in np.arange(clusters[1]):
#         cluster_number = a + 1
#         cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
#     largest_cluster = np.argmax(cluster_size) + 1
#     center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
#     mslp = ds_sfc['prmsl'].where(clusters[0] == largest_cluster).min()

#     center_lat = ds_sfc.latitude.isel(latitude =int(np.round(center_y)))
#     center_lon = ds_sfc.longitude.isel(longitude =int(np.round(center_x)))
    
#     center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
#     return center_info

def random_link_selection(num_of_frames, links_path):
    result = []
    with open(links_path) as f:
        links = pd.read_csv(f)
    random_frames = links.sample(num_of_frames)
    for link in random_frames.values:
        result.append(('https://noaa-nws-hafs-pds.s3.amazonaws.com/' + link).squeeze())
    
    return result
    

In [3]:
atm_args = dict(typeOfKey = 'isobaricInhPa',filename="Model_Data/temp_gribs/atm_temp.grib2" )
sfc_args = dict(typeOfKey = 'meanSea',filename="Model_Data/temp_gribs/sfc_temp.grb2" )

In [4]:
random_links = random_link_selection(8, 'Model_Data/links/link_list.txt')

In [5]:
with PdfPages('Storm_CentersNewAtm.pdf') as pdf:
    for link in random_links:
        sfc_ds = utils.download_and_open(link, **sfc_args)
        sfc_ds['prmsl'] = sfc_ds['prmsl']/100
        try:
            print('Getting surface center.')
            sfc_center, center_cluster, lat_nest, lon_nest = utils.get_sfc_center(sfc_ds)
        except Exception as e:
            print(e)
            continue
        if not sfc_center:
            continue

        ## Plotting

        fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (10,8), layout="constrained", subplot_kw = dict(projection = ccrs.PlateCarree()))
        ax[1].contour(lon_nest, lat_nest, center_cluster, colors = 'red')
        zoomed_in_cf = ax[1].contourf(lon_nest, lat_nest, sfc_ds['prmsl'].sel(latitude = lat_nest, longitude = lon_nest).values, cmap = 'Blues_r')
        ax[1].scatter(sfc_center['center_lon'], sfc_center['center_lat'], color = 'red', marker = 'x')
        ax[1].add_feature(cfeature.COASTLINE)
        ax[1].gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

        ax[0].contourf(sfc_ds.longitude.values, sfc_ds.latitude.values, sfc_ds['prmsl'].values,  cmap = 'Blues_r')
        ax[0].contour(lon_nest, lat_nest, center_cluster)
        ax[0].add_feature(cfeature.COASTLINE)
        ax[0].gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

        fig.colorbar(zoomed_in_cf, location = 'bottom', ax=[ax[0], ax[1]], label = 'Minimum Central Pressure (mb)')
        rounded_mslp = np.round(sfc_center['mslp'],decimals=0)
        fig.suptitle(f'Minimum Central Pressure {rounded_mslp}',y=0.75)
        pdf.savefig()
        plt.close()


        atm_ds = utils.download_and_open(link, **atm_args)

        center_points = get_atm_center(atm_ds)
        print(center_points)
        # atm_ds = atm_ds.sel(isobaricInhPa = [sfc_center['mslp'], 700.0, 200.0], method = 'nearest')
        # current_level_center = sfc_center
        # valid_levels = True
        # for level in atm_ds.isobaricInhPa:
        #     print(f'Processing {level}mb level')
        #     atm_ds_lvl = atm_ds.sel(isobaricInhPa = level)
        #     current_level_center = get_atm_center(atm_ds_lvl, current_level_center)
        #     if not current_level_center:
        #         valid_levels = False
        #         break
        #     if not valid_levels:
        #         continue
        #     atm_ds_lvl['gh'].plot()
        #     plt.show()

Ignoring index file 'Model_Data/temp_gribs/sfc_temp.grb2.5b7b6.idx' older than GRIB file


Getting surface center.
1005.8400268554688mb is above the pressure threshold of 1005mb


KeyboardInterrupt: 

In [ ]:
sfc_ds = utils.download_and_open(random_links[0], **sfc_args)
sfc_ds['prmsl'] = sfc_ds['prmsl']/100

In [ ]:
sfc_center = get_sfc_center(sfc_ds)

In [ ]:
atm_ds = utils.download_and_open(random_links[0], **atm_args)
atm_ds = atm_ds.sel(isobaricInhPa = [sfc_center['mslp'].values, 700.0, 200.0], method = 'nearest')